In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import os

# ================= 配置区 =================
file_path = "/mnt/data/zwl/verl/data/mixed.jsonl"
# 替换为你实际使用的 Tokenizer 路径，例如你之前的 "/mnt/oss/zwl/checkpoints/qwen3_4b_sft_edge_only/global_step_124/huggingface"
tokenizer_name_or_path = "Qwen/Qwen2.5-7B-Instruct" 

# ================= 加载 Tokenizer =================
print(f"正在加载 Tokenizer: {tokenizer_name_or_path}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path, trust_remote_code=True)
except Exception as e:
    print(f"Tokenizer 加载失败，请检查路径或网络: {e}")
    # 降级方案：如果无法加载，使用粗略的字符长度代替（中文约1字符=0.6~1token，英文约1单词=1.3token）
    tokenizer = None

# ================= 统计逻辑 =================
global_lengths = []
category_data = []

print(f"开始解析文件: {file_path}")
if not os.path.exists(file_path):
    print("文件不存在，请检查路径！")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        # 统计总行数用于进度条
        total_lines = sum(1 for _ in f)
        f.seek(0)
        
        for line in tqdm(f, total=total_lines, desc="Processing JSONL"):
            if not line.strip():
                continue
                
            data = json.loads(line)
            output_text = data.get("output", "")
            view = data.get("view", "unknown")
            difficulty = data.get("difficulty", "unknown")
            
            # 计算 Token 长度
            if tokenizer:
                token_length = len(tokenizer.encode(output_text))
            else:
                token_length = len(output_text) # 降级为字符长度
                
            global_lengths.append(token_length)
            category_data.append({
                "view": view,
                "difficulty": difficulty,
                "length": token_length
            })

# ================= 数据汇总与展示 =================
if global_lengths:
    df = pd.DataFrame(category_data)
    
    # 计算全局平均
    global_avg = sum(global_lengths) / len(global_lengths)
    global_max = max(global_lengths)
    global_min = min(global_lengths)
    
    print("\n" + "="*50)
    print("🎯 全局 Output Token 长度统计")
    print("="*50)
    metric_name = "Token 数量" if tokenizer else "字符数量 (Fallback)"
    print(f"总样本数: {len(global_lengths)}")
    print(f"平均 {metric_name}: {global_avg:.2f}")
    print(f"最大 {metric_name}: {global_max}")
    print(f"最小 {metric_name}: {global_min}")
    
    # 按照 view 和 difficulty 聚合
    print("\n" + "="*50)
    print("📊 细分维度统计 (View & Difficulty)")
    print("="*50)
    
    # 使用 pivot_table 制作直观的交叉表
    summary_df = df.pivot_table(
        index=["view", "difficulty"], 
        values="length", 
        aggfunc=["mean", "count", "max", "min"]
    )
    
    # 重命名列名，使其在 Notebook 中更易读
    summary_df.columns = ["平均长度", "样本数量", "最大长度", "最小长度"]
    summary_df["平均长度"] = summary_df["平均长度"].round(2)
    
    # 在 Jupyter Notebook 中直接展示表格
    display(summary_df)
else:
    print("未提取到有效数据。")

正在加载 Tokenizer: Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

开始解析文件: /mnt/data/zwl/verl/data/mixed.jsonl


Processing JSONL:   0%|          | 0/1000 [00:00<?, ?it/s]


🎯 全局 Output Token 长度统计
总样本数: 1000
平均 Token 数量: 1033.94
最大 Token 数量: 6922
最小 Token 数量: 136

📊 细分维度统计 (View & Difficulty)


平均长度  样本数量  最大长度  最小长度
view difficulty                           
edge hard        1224.99   318  6922   342
     medium       937.44    18  1734   550
     simple       789.49   264  2061   136
path hard        1245.37   212  5131   276
     medium       895.50    12  1374   525
     simple       820.03   176  1905   136